# Calculus, Gradients, and Optimization
Run the cells in order using the play button or Shift+Enter. You will use the same fictional student throughout. First compare two losses using subtraction and division. After you can explain the hand update, ask PyTorch to check it.

## Ava’s prediction was 55. Suppose she scored 60.
Use the rule from Module 01. Ava studied two hours and completed three sets. Her fictional exam score is now known.

2 × 10 + 3 × 5 + 20 = 55

The inputs and observed score stay on the page. We can edit the two weights and the starting score, called the bias. First predict what would happen if only the bias increased by one.

**Before running the code:** What would the model predict with bias 21? What would Ava’s observed score be?

In [ ]:
import torch
x = torch.tensor([2., 3.], dtype=torch.float64)
w = torch.tensor([10., 5.], dtype=x.dtype)
bias = 20.
mean = x @ w + bias
print(mean.item(), (x @ w + 21).item())

**Worked answer.** The prediction would rise to 56. Her observed exam score stays 60. We changed the rule, not the record.

## Which mean makes 60 more plausible?
Recall the last probability lesson. Compare two Gaussian score distributions with the same spread of 10 points. Center one at 55 and one at 56.

Loss at 55 ≈ 3.346524      Loss at 56 ≈ 3.301524

The bell centered at 56 is higher at the observed 60. Module 02 called minus the natural log of that height the negative log-likelihood, or loss. A better fit here has a smaller loss.

**Before running the code:** If we shift the mean from 55 down to 54, should the loss rise or fall?

In [ ]:
import torch
def loss(mean):
    mean = torch.tensor(mean, dtype=torch.float64)
    model = torch.distributions.Normal(mean, 10.)
    return -model.log_prob(mean.new_tensor(60.))

for mean in [54, 55, 56]:
    print(mean, round(loss(mean).item(), 6))

**Worked answer.** It rises. At the same spread, 60 is farther from the center 54, so the density there is smaller.

## Calculate the part of the loss that changes
Keep the spread fixed. Computing the Gaussian loss gives a fixed starting amount plus a penalty for missing 60. The symbol μ names the predicted mean.

Penalty = (μ − 60)² ÷ (2 × 10²) = (μ − 60)² ÷ 200

At 55 the signed miss is −5. Squaring gives 25, and dividing by 200 gives 0.125. Add the same fixed amount, about 3.221524, to recover the full loss. The notebook shows where both terms come from.

**Before running the code:** At a mean of 58, what penalty do we add?

In [ ]:
import torch
def penalty(mean):
    return (mean - 60.) ** 2 / 200.

means = torch.tensor([55., 56., 58., 60.])
print(penalty(means))

**Worked answer.** 58 − 60 = −2. Square it and divide by 200 to get 4/200 = 0.020.

## A slope answers “how much per point?”
Move the mean from 55 to 56. Compute the change in the penalty, then divide by the change in the mean. Use after minus before in both places.

Average slope = (0.080 − 0.125) / (56 − 55) = −0.045

Read this as “the loss fell by 0.045 per point of mean over this move.” The fixed amount cancels in the subtraction. This calculation uses two distinct predictions.

**Before running the code:** From 55 to 57, the penalty changes from 0.125 to 0.045. What is the average slope?

In [ ]:
import torch
def penalty(mean):
    return (mean - 60.) ** 2 / 200.

m = torch.tensor(55., dtype=torch.float64)
print((penalty(m+2) - penalty(m)) / 2)

**Worked answer.** The change is −0.080 over two score points. Divide by 2 to get −0.040 per point.

## Shrink the measuring step
The curve bends, so the rate across two points can differ from the rate right at 55. Start at 55 each time. Call the trial move h.

average slope = [penalty(55 + h) − penalty(55)] / h

The values approach −0.050 as h approaches zero. This limiting slope is the derivative at 55. On a graph it is the slope of the line that follows the curve’s direction right there, called the tangent.

**Before running the code:** Why do we shrink h instead of setting h equal to zero in the fraction?

In [ ]:
import torch
def penalty(mean):
    return (mean - 60.) ** 2 / 200.

m = torch.tensor(55., dtype=torch.float64)
for h in [1., .1, .01]:
    print(h, ((penalty(m+h) - penalty(m))/h).item())

**Worked answer.** At zero both differences are zero and division by zero is undefined. We examine the ratios for nonzero steps and see which value they approach.

## Find the exact slope with ordinary multiplication
At 55 the miss is −5. After a trial move h the miss is −5 + h. Multiply this expression by itself before taking the difference.

(−5 + h)(−5 + h) = 25 − 5h − 5h + h² = 25 − 10h + h²

Subtract the old square, 25. Divide by 200h. For nonzero h, (−10h + h²)/(200h) = (−10 + h)/200. Now let h approach zero. The result is −10/200 = −0.05.

**Before running the code:** Where did the two copies of −5h come from?

In [ ]:
import torch
def penalty(mean):
    return (mean - 60.) ** 2 / 200.

h = torch.tensor(0.1, dtype=torch.float64)
print(((penalty(55+h)-penalty(55))/h).item(), ((-10+h)/200).item())

**Worked answer.** Each factor contributes one cross product: (−5) × h and h × (−5). Together they give −10h.

## Use the same reasoning anywhere on the curve
Call the current signed miss r = μ − 60. Replacing −5 with r gives (r + h)² − r² = 2rh + h². Divide by 200h and shrink h.

dL/dμ = 2r/200 = (μ − 60)/100

Read dL/dμ as “change in loss per point of predicted mean, right here.” A negative slope means a small move to the right lowers loss. At 60 the slope vanishes and the squared penalty reaches its minimum.

**Before running the code:** At 58, which way should the mean move, and what is its local slope?

For another observed score y and any fixed positive spread σ, the same cancellation gives (μ − y)/σ². For Ava, y = 60 and σ² = 100.

In [ ]:
import torch
def penalty(mean):
    return (mean - 60.) ** 2 / 200.

means = torch.tensor([55., 58., 60., 65.])
print((means - 60)/100)

**Worked answer.** (58 − 60)/100 = −0.02. A small increase toward 60 reduces the loss.

## Move just one weight
Return to the original model. Change the hours weight from 10 to 10.10. Leave the sets weight at 5 and bias at 20.

New mean = 2 × 10.10 + 3 × 5 + 20 = 55.20

A weight increase of 0.10 moves the mean by 0.20 because Ava has two hours. Try the same change separately on the sets weight or on the bias. Always reset first.

**Before running the code:** If only the sets weight rose by 0.20, how far would the mean move?

In [ ]:
import torch
x = torch.tensor([2., 3.], dtype=torch.float64)
w = torch.tensor([10., 5.], dtype=x.dtype)
bias = 20.
mean = x @ w + bias
trial = w.clone()
trial[1] += .2
print((x @ trial + bias).item())

**Worked answer.** There are three sets, so the mean would move by 3 × 0.20 = 0.60 points, from 55 to 55.60.

## Follow the change from one weight to the loss
An hours-weight change acts in two stages. It moves the mean, then the new mean changes the loss. Multiply the rates along that route. This multiplication is the chain rule.

(−0.05 loss per mean point) × (2 mean points per unit of hours weight) = −0.10

Holding the other parameters fixed makes this a partial derivative. We write ∂L/∂w₁ for the loss slope with respect to the first weight. For a tiny increase of 0.01, expect a loss change of about −0.001.

**Before running the code:** Why is −0.10 a rate, while −0.001 is a predicted change?

In [ ]:
import torch
def loss(mean):
    mean = torch.tensor(mean, dtype=torch.float64)
    model = torch.distributions.Normal(mean, 10.)
    return -model.log_prob(mean.new_tensor(60.))

actual = loss(55 + 2 * .01) - loss(55)
print("Estimated:", -.1 * .01, "Actual:", actual.item())

**Worked answer.** −0.10 is loss change per unit of weight near the starting model. Multiply by the actual move, 0.01, to estimate the change in loss.

## Collect the three partial derivatives
Repeat the calculation for the sets weight and the bias. Keep the order hours weight, sets weight, bias. The ordered list of partial derivatives is the gradient.

gradient = [−0.05 × 2, −0.05 × 3, −0.05 × 1] = [−0.10, −0.15, −0.05]

The second entry has the largest magnitude because the mean changes by three points per unit change in that weight for Ava. This comparison depends on the input and its units.

**Before running the code:** If a different student had zero practice sets, what would that student’s loss slope be for the sets weight?

In [ ]:
import torch
x = torch.tensor([2., 3.], dtype=torch.float64)
w = torch.tensor([10., 5.], dtype=x.dtype)
bias = 20.
mean = x @ w + bias
d_mean = (mean - 60)/100
print("Weights:", d_mean * x, "Bias:", d_mean)

**Worked answer.** Zero. Changing the sets weight cannot affect this student’s prediction because that weight is multiplied by zero.

## Turn slopes into one update
The slopes tell us how loss responds. The learning rate sets the size of a training step. Choose 1 for this example. Subtract rate × slope from each starting parameter.

new value = old value − learning rate × partial derivative

For the hours weight, 10 − 1 × (−0.10) = 10.10. Subtracting a negative raises the weight, which raises Ava’s mean toward 60. Use slopes from the same starting prediction for all three changes.

**Before running the code:** At a rate of 0.5, what would the new hours weight be?

In [ ]:
import torch
x = torch.tensor([2., 3.], dtype=torch.float64)
w = torch.tensor([10., 5.], dtype=x.dtype)
bias = 20.
mean = x @ w + bias
rate = 1.
slope = (mean - 60)/100
print(w - rate * slope * x, bias - rate * slope)

**Worked answer.** 10 − 0.5 × (−0.10) = 10.05. Halving the rate halves this parameter change.

## Recompute the prediction and the loss
Use Module 01’s rule again with the three updated values. Then use Module 02’s loss to check whether the update helped.

2 × 10.10 + 3 × 5.15 + 20.05 = 55.70

The hours term adds 0.20, the sets term adds 0.45 and the bias adds 0.05. The miss shrinks from 5 to 4.30 points. The loss falls from 3.346524 to 3.313974.

**Before running the code:** Why did the sets-weight increase of 0.15 add 0.45 to the prediction?

In [ ]:
import torch
def loss(mean):
    mean = torch.tensor(mean, dtype=torch.float64)
    model = torch.distributions.Normal(mean, 10.)
    return -model.log_prob(mean.new_tensor(60.))

updated = 2*10.10 + 3*5.15 + 20.05
print(updated, loss(updated).item())

**Worked answer.** Ava completed three sets, so a change of 0.15 points per set adds 3 × 0.15 = 0.45 points.

## Show PyTorch what can change
Now verify the work you did by hand. Give PyTorch the same two weights and bias. The flag requires_grad=True tells it to track how calculations depend on each parameter.

forward pass: weights and bias → predicted mean → loss

The input vector, observed score, and spread are fixed. Run the model forward once. At the end, loss holds one number and PyTorch has recorded the operations that produced it.

**Before running the code:** Which three numerical derivatives should PyTorch produce for the starting model?

In [ ]:
import torch
x = torch.tensor([2., 3.], dtype=torch.float64)
w = torch.tensor([10., 5.], dtype=x.dtype, requires_grad=True)
b = torch.tensor(20., dtype=x.dtype, requires_grad=True)
mean = x @ w + b
observed = x.new_tensor(60.)
loss = -torch.distributions.Normal(mean, 10.).log_prob(observed)
print(mean.item(), loss.item())

**Worked answer.** In parameter order they should be −0.10, −0.15 and −0.05. Write them down before continuing.

## Read the gradients before changing anything
Call loss.backward() to trace the recorded operations backward using the chain rule. PyTorch stores the two weight slopes in w.grad and the bias slope in b.grad.

w.grad ≈ [−0.10, −0.15]     b.grad ≈ −0.05

Calling backward calculates slopes. A separate subtraction changes the parameters. The notebook checks each slope against the hand calculation and verifies the updated prediction and loss.

**Before running the code:** Immediately after backward(), before any subtraction, what is the hours weight?

In [ ]:
import torch
x = torch.tensor([2., 3.], dtype=torch.float64)
w = torch.tensor([10., 5.], dtype=x.dtype, requires_grad=True)
b = torch.tensor(20., dtype=x.dtype, requires_grad=True)
mean = x @ w + b
observed = x.new_tensor(60.)
loss = -torch.distributions.Normal(mean, 10.).log_prob(observed)
loss.backward()
expected_w = x.new_tensor([-0.10, -0.15])
expected_b = x.new_tensor(-0.05)
torch.testing.assert_close(w.grad, expected_w)
torch.testing.assert_close(b.grad, expected_b)
print(w.grad.tolist(), b.grad.item())

**Worked answer.** It is still 10. Its gradient is now stored as −0.10. Only the update changes the weight to 10.10.

### Apply the verified gradients
Run the preceding cell first. Set the learning rate to 1. Use no_grad() while subtracting because this subtraction sets new parameter values. Recompute the mean and loss using the updated parameters.

In [ ]:
with torch.no_grad():
    w -= 1.0 * w.grad
    b -= 1.0 * b.grad
    updated_mean = x @ w + b
    updated_loss = -torch.distributions.Normal(updated_mean, 10.).log_prob(observed)
torch.testing.assert_close(updated_mean, x.new_tensor(55.70))
torch.testing.assert_close(updated_loss, x.new_tensor(3.3139736261987185))
print(f"Mean {updated_mean.item():.2f}; loss {updated_loss.item():.6f}")

## Try a smaller step and a larger one
Restart at 55 each time. At rate 1 the mean gains 0.70. At rate 10 all three parameter changes are ten times as large, so the mean gains 7.00.

rate 1: 55 → 55.70      rate 10: 55 → 62      rate 16: 55 → 66.20

Compare the final distance to 60 with the original distance of five. Crossing 60 can still help. Going all the way to 66.20 makes the miss larger.

**Before running the code:** Restart at 55 with rate 5. What is the mean after one update? Does loss fall?

In [ ]:
import torch
def loss(mean):
    mean = torch.tensor(mean, dtype=torch.float64)
    model = torch.distributions.Normal(mean, 10.)
    return -model.log_prob(mean.new_tensor(60.))

for rate in [1, 5, 10, 16]:
    new_mean = 55 + rate * .70
    print(rate, new_mean, loss(new_mean).item())

**Worked answer.** 55 + 5 × 0.70 = 58.50. It is 1.50 points from 60, closer than before, so loss falls.

## What did one observation teach us?
For Ava, every parameter combination that predicts 60 reaches the same lowest loss. A different student can receive a different prediction from those same models.

2w₁ + 3w₂ + b = 60   ⇒   ∇L = 0

In Module 04 we will ask whether a rule fitted to training examples predicts held-out observations well. For the in-class variant, explain one update with unfamiliar values.

**Before running the code:** Can two models fit Ava exactly yet disagree for a student with one hour and one practice set?

In [ ]:
import torch
X = torch.tensor([[2., 3.], [1., 1.]])
for w, b in [([10., 5.], 25.), ([10., 6.], 22.)]:
    print((X @ torch.tensor(w) + b).tolist())

**Worked answer.** Yes. Weights [10, 5] with bias 25 and weights [10, 6] with bias 22 both give Ava 60. For [1, 1] they predict 40 and 38 respectively.

## Further reading: where the fixed amount comes from
The Gaussian density at the observed score is exp(−penalty)/(σ√(2π)). Here exp is the exponential function. Taking its natural logarithm undoes the exponential, and the logarithm of a quotient subtracts the logarithm of the denominator. So log(density) = −penalty − log(σ√(2π)). Reversing the sign gives the NLL used in the lesson. With σ = 10, the log term stays fixed. Run the cell to compare the expanded expression with PyTorch’s distribution.

In [ ]:
import torch
def loss(mean):
    mean = torch.tensor(mean, dtype=torch.float64)
    model = torch.distributions.Normal(mean, 10.)
    return -model.log_prob(mean.new_tensor(60.))

import math
fixed = math.log(10 * math.sqrt(2*math.pi))
for m in [55., 56., 60.]:
    print(m, fixed + (m-60)**2 / 200, loss(m).item())